In [5]:
%pip install pytubefix 


Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
from pytubefix import YouTube

In [7]:
link = 'https://www.youtube.com/watch?v=bByJ2CzkBlc&list=RDbByJ2CzkBlc&start_radio=1'

In [8]:
yt = YouTube(link)
print(yt)
print(f"Titulo: {yt.title}")
print(f"Número de vistas: {yt.views}")
print(f"largo del video {yt.length}")
print(f"Minuatura del video {yt.thumbnail_url}")

<pytubefix.__main__.YouTube object: videoId=bByJ2CzkBlc>
Titulo: Separados - Hombres G - Letra
Número de vistas: 71119
largo del video 246
Minuatura del video https://i.ytimg.com/vi/bByJ2CzkBlc/sddefault.jpg


In [9]:
print(yt.streams)#resuluciones del video

[<Stream: itag="18" mime_type="video/mp4" res="360p" fps="30fps" vcodec="avc1.42001E" acodec="mp4a.40.2" progressive="True" sabr="False" type="video">, <Stream: itag="136" mime_type="video/mp4" res="720p" fps="30fps" vcodec="avc1.4d401f" progressive="False" sabr="False" type="video">, <Stream: itag="134" mime_type="video/mp4" res="360p" fps="30fps" vcodec="avc1.4d401e" progressive="False" sabr="False" type="video">, <Stream: itag="160" mime_type="video/mp4" res="144p" fps="30fps" vcodec="avc1.4d400c" progressive="False" sabr="False" type="video">, <Stream: itag="139" mime_type="audio/mp4" abr="48kbps" acodec="mp4a.40.5" progressive="False" sabr="False" type="audio">, <Stream: itag="140" mime_type="audio/mp4" abr="128kbps" acodec="mp4a.40.2" progressive="False" sabr="False" type="audio">, <Stream: itag="251" mime_type="audio/webm" abr="160kbps" acodec="opus" progressive="False" sabr="False" type="audio">]


In [10]:
print(yt.streams.get_highest_resolution())#resolucion mas alta

<Stream: itag="18" mime_type="video/mp4" res="360p" fps="30fps" vcodec="avc1.42001E" acodec="mp4a.40.2" progressive="True" sabr="False" type="video">


In [11]:
ys = yt.streams.get_highest_resolution()
print('Descargando...')
ys.download()
print('Descarga completa')

Descargando...
Descarga completa


In [12]:
audio = yt.streams.filter(only_audio=True).first()
print('Descargando Audio')
outputFile = audio.download()
print('Descarga completa')

Descargando Audio
Descarga completa


In [13]:
basename = os.path.basename(outputFile)
print(basename)

Separados - Hombres G - Letra.m4a


In [14]:
nombre, formato = basename.split('.')
print(f"Nombre: {nombre} - Formato: {formato}")

Nombre: Separados - Hombres G - Letra - Formato: m4a


In [15]:
audioFile = f"{nombre}.mp3"
print(audioFile)

Separados - Hombres G - Letra.mp3


In [16]:
audioFile = audioFile.replace(" ", "_")
print(audioFile)

Separados_-_Hombres_G_-_Letra.mp3


In [18]:
os.rename(basename, audioFile)

FileExistsError: [WinError 183] No se puede crear un archivo que ya existe: 'Separados - Hombres G - Letra.m4a' -> 'Separados_-_Hombres_G_-_Letra.mp3'

In [19]:
def youtubeAudioDescargador(link):
    import os
    from pytubefix import YouTube
    yt = YouTube(link)
    audio = yt.streams.filter(only_audio=True).first()
    print('Descargando Audio')
    outputFile = audio.download()
    if os.path.exists(outputFile):
            print('Descarga completa')
    else:
            print("Error de descarga")
            return False
    
    basename = os.path.basename(outputFile)
    nombre, formato = basename.split('.')
    audioFile = f"{nombre}.mp3"
    audioFile = audioFile.replace(" ", "_")
    os.rename(basename, audioFile)
    return audioFile

In [20]:
mp3Archivo = youtubeAudioDescargador(input(f"Link de youtube: "))

Descargando Audio
Descarga completa


FileExistsError: [WinError 183] No se puede crear un archivo que ya existe: 'Separados - Hombres G - Letra.m4a' -> 'Separados_-_Hombres_G_-_Letra.mp3'

In [21]:
from openai import OpenAI
key = ""
cliente = OpenAI(api_key = key)

In [22]:
def transcribir(audioFile):
    import os
    from openai import OpenAI
    key = ""
    cliente = OpenAI(api_key = key)
    with open(audioFile, 'rb') as audio:
        print("Comenzando la transcripcion...", end="")
        transcripcion = cliente.audio.transcriptions.create(file=audio, model="whisper-1")
        print("Terminado")

        nombre, extension = os.path.splitext(audioFile)
        transcripcionNombre = f"transcripcion-{nombre}.txt"
        with open(transcripcionNombre, "w") as file:
            file.write(transcripcion.text)
    
    return transcripcionNombre

In [23]:
transcripcionMp3 = transcribir(mp3Archivo)
with open(transcripcionMp3) as f:
    print(f.read())

NameError: name 'mp3Archivo' is not defined

In [ ]:
def resumen(transcripcionMp3):
    with open(transcripcionMp3) as file:
        transcripcion = file.read()

    systemPromt = "Experto en resumir textos"
    prompt = f"""Crea un resumen para el siguiente texto: {transcripcion}

    añade un titulo al resumen
    el resumen deberia cubrir los mas importante del texto
    """

    respuesta = cliente.chat.completions.create(
    model = "gpt-4",
    messages = [
        {'role': 'system', 'content': systemPromt},
        {'role': 'user', 'content': prompt}
    ],
    temperature = 1.3,
    max_tokens = 2048
    )

    r = respuesta.choices[0].message.content

    return r

In [ ]:
print(resumen(transcripcionMp3))